In [1]:

import numpy as np
import sys
%load_ext autoreload
%autoreload 2
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent)) 
from commom_utils.systems import *
from commom_utils.ode_system import ODESystem, check_system_ok, SyntheticDataGenerator
from commom_utils.system_config import create_system, SYSTEM_CONFIGS
from gauss_newton.utils import plot_solution
import matplotlib.pyplot as plt
from gauss_newton.problem import MultipleShooting
from gauss_newton.adaptive import run_optimization_adaptive
from scipy.interpolate import interp1d
from experiments.data_utils import  LogReaderV2, create_jax_interpolator

In [2]:
root_dir = Path("/home/iachichkanov/autotech/GaussNewton/experiments/voyax_free/Lateral Dynamics/80 kph")
# root_dir = Path("/home/iachichkanov/autotech/GaussNewton/experiments/voyax_free/Skipad/Left")

data_storage_EPS_angle = LogReaderV2(root_dir/"EPS_angle.csv")
data_storage_ESP_IMU = LogReaderV2(root_dir/"ESP_IMU.csv")
data_storage_IMU_2 = LogReaderV2(root_dir/"IMU_2.csv")
data_storage_wheel_speed = LogReaderV2(root_dir/"ESP_1_rear_speed_wheel.csv")

Загружен default: 18024 строк из /home/iachichkanov/autotech/GaussNewton/experiments/voyax_free/Lateral Dynamics/80 kph/EPS_angle.csv
Загружен default: 9012 строк из /home/iachichkanov/autotech/GaussNewton/experiments/voyax_free/Lateral Dynamics/80 kph/ESP_IMU.csv
Загружен default: 6008 строк из /home/iachichkanov/autotech/GaussNewton/experiments/voyax_free/Lateral Dynamics/80 kph/IMU_2.csv
Загружен default: 6008 строк из /home/iachichkanov/autotech/GaussNewton/experiments/voyax_free/Lateral Dynamics/80 kph/ESP_1_rear_speed_wheel.csv


In [3]:
data_storage_EPS_angle.add_batch( "EPS_SAS_SteerAngleValid", use_jax_interp=1, scale_coef = np.deg2rad(1))
data_storage_ESP_IMU.add_batch( "LattAx", use_jax_interp=1, scale_coef =-1)
data_storage_ESP_IMU.add_batch( "YAW_Rate", use_jax_interp=1,  scale_coef = -np.deg2rad(1))
data_storage_IMU_2.add_batch( "YAW_Rate", use_jax_interp=1,  scale_coef = -np.deg2rad(1))
data_storage_wheel_speed.add_batch("IPB_wheelSpeedRL", use_jax_interp=1, scale_coef = 1.0/3.6)
data_storage_wheel_speed.add_batch("IPB_wheelSpeedRR", use_jax_interp=1, scale_coef = 1.0/3.6)

data_storage_EPS_angle.process_all()
data_storage_ESP_IMU.process_all()
data_storage_IMU_2.process_all()
data_storage_wheel_speed.process_all()
#t = data_storage_EPS_angle.get_time("EPS_SAS_SteerAngleValid")
f_steer = data_storage_EPS_angle.get_f_interp("EPS_SAS_SteerAngleValid")
f_lat_ax = data_storage_ESP_IMU.get_f_interp("LattAx")
f_yaw_rate1 = data_storage_ESP_IMU.get_f_interp("YAW_Rate")
f_yaw_rate2 = data_storage_IMU_2.get_f_interp("YAW_Rate")
f_wheelSpeedRL = data_storage_wheel_speed.get_f_interp("IPB_wheelSpeedRL")
f_wheelSpeedRR = data_storage_wheel_speed.get_f_interp("IPB_wheelSpeedRR")


Добавлен в очередь EPS_SAS_SteerAngleValid из default: time [1780935450.234, 1780935510.305], values [-2.272, 1.787]
Добавлен в очередь LattAx из default: time [1780935450.236, 1780935510.294], values [-8.020, 7.890]
Добавлен в очередь YAW_Rate из default: time [1780935450.236, 1780935510.294], values [-0.550, 0.509]
Добавлен в очередь YAW_Rate из default: time [1780935450.236, 1780935510.294], values [-0.544, 0.513]
Добавлен в очередь IPB_wheelSpeedRL из default: time [1780935450.233, 1780935510.292], values [9.629, 21.700]
Добавлен в очередь IPB_wheelSpeedRR из default: time [1780935450.233, 1780935510.292], values [9.551, 21.778]

Общий t0 = 1780935450.234207
  используем jax
Обработан EPS_SAS_SteerAngleValid: норм. время [0.000, 60.070]
Общий временной интервал: [0.000, 60.070]

Общий t0 = 1780935450.236109
  используем jax
Обработан LattAx: норм. время [0.000, 60.058]
  используем jax
Обработан YAW_Rate: норм. время [0.000, 60.058]
Общий временной интервал: [0.000, 60.058]

Общий 

In [4]:
[data_storage_EPS_angle.t2, data_storage_ESP_IMU.t2, data_storage_IMU_2.t2]

[np.float64(60.07045912742615),
 np.float64(60.05753779411316),
 np.float64(60.05738306045532)]

In [5]:
t2 = np.min([data_storage_EPS_angle.t2, data_storage_ESP_IMU.t2, data_storage_IMU_2.t2])
t1 = np.max([data_storage_EPS_angle.t1, data_storage_ESP_IMU.t1, data_storage_IMU_2.t1])
t = np.linspace(t1, t2 - 1, 8000)

In [6]:
t1

np.float64(0.0)

In [7]:
vx_data = (f_wheelSpeedRL(t) + f_wheelSpeedRR(t))/2
if(0):
    f_vx =  interp1d(t, vx_data, kind = 'linear') 
else:
    f_vx = create_jax_interpolator(t, vx_data, method='linear')

In [8]:
wheelbase = 2.96
gear_ratio_test  = 14.63
rwa = f_steer(t)/gear_ratio_test
vx = f_vx(t)
plt.plot(t,  vx*np.tan(rwa)/(wheelbase))
plt.plot(t, f_yaw_rate1(t))
#plt.plot(t, vx)
# plt.plot(t, f_steer(t))

# plt.plot(t, f_wheelSpeedRL(t))
# plt.plot(t, f_wheelSpeedRR(t))
# plt.plot(t, f_vx(t))
# plt.plot(t, f_yaw_rate1(t))
# plt.plot(t, f_yaw_rate2(t))

<Figure size 640x480 with 1 Axes>

In [9]:

plt.plot(t, vx*vx*rwa/wheelbase)
plt.plot(t, f_lat_ax(t))

<Figure size 640x480 with 1 Axes>

In [10]:


def get_input_signals(t):
    steering = f_steer(t)
    Gr = 1.46367113e+01
    Gr_sq = 3.98385740e-02
    offset =  -3.02111275e-03
    rwa = (steering + offset)/(Gr - Gr_sq*(steering + offset)**2) 
    return [f_vx(t), rwa]      



class BicycleModelNormed(ODESystem):
    """
    Безразмерная модель одноколейного транспортного средства.
    Состояния: [vy, r]  (поперечная скорость, угловая скорость рыскания)
    Входы:     [vx, steering]  (продольная скорость, угол руля)
    Параметры (theta):
        theta[0] : Cf_norm = Cf / (m*g)        # ~5..12
        theta[1] : Cr_norm = Cr / (m*g)        # ~5..12
        theta[2] : a_rel   = a / L             # ~0.35..0.55
        theta[3] : Iz_norm = Iz / (m*L*L)      # ~0.18..0.30
        theta[4] : steer_ratio                 # ~14..18
    """
    def __init__(self, m, L, g=9.81):
        self.m = m                  # масса, кг
        self.L = L                  # колёсная база, м
        self.g = g                  # ускорение свободного падения
        super().__init__(2, 4, 2)   # 2 состояния, 5 параметров, 2 входа
    
    def get_lateral_forces(self, state, theta, u):
        vy, r = state[0], state[1]
        vx, delta = u[0], u[1]

        # Извлекаем безразмерные параметры
        Cf_norm = theta[0]
        Cr_norm = theta[1]
        a_rel   = theta[2]
        Iz_norm = theta[3]
        #offset = theta[5]

        # Восстанавливаем физические величины
        Cf = Cf_norm * self.m * self.g
        Cr = Cr_norm * self.m * self.g
        a  = a_rel * self.L
        b  = self.L - a

        # Малые углы бокового увода
        alpha_f = delta - (vy + a * r) / vx
        alpha_r = -(vy - b * r) / vx

        # Боковые силы
        Fyf = Cf * alpha_f
        Fyr = Cr * alpha_r
        return Fyf, Fyr

    

    def get_derivative(self, state, theta, u):
        vy, r = state[0], state[1]
        vx, steering_wheel = u[0], u[1]

        # Извлекаем безразмерные параметры
        a_rel   = theta[2]
        Iz_norm = theta[3]
        #offset = theta[5]
        Fyf, Fyr = self.get_lateral_forces(state, theta, u)
        a  = a_rel * self.L
        b  = self.L - a
        Iz = Iz_norm * self.m * self.L * self.L
        # Уравнения движения
        dvy = (Fyf + Fyr) / self.m - vx * r
        dr  = (a * Fyf - b * Fyr) / Iz

        return ca.vertcat(dvy, dr)
    
    def observation(self, state: SX, theta: SX, u: SX):
        Fyf, Fyr = self.get_lateral_forces(state, theta, u)
        a_lat = (Fyf + Fyr)/self.m
        vy, r = state[0], state[1]
        return ca.vertcat(a_lat, r)
    
class BicycleModelWithMass(ODESystem):
    """
    Модель одноколейного ТС с идентифицируемой массой.
    Состояния: [vy, r]  (поперечная скорость, угловая скорость рыскания)
    Входы:     [vx, steering_wheel]
    Параметры (theta):
        theta[0] : m_rel    = m / m_nom
        theta[1] : Cf_rel   = Cf / (m_nom * g)
        theta[2] : Cr_rel   = Cr / (m_nom * g)
        theta[3] : a_rel    = a / L
        theta[4] : Iz_rel   = Iz / (m * L^2)   (коэффициент формы)
        theta[5] : steer_ratio
    """
    def __init__(self, m_nom, L, g=9.81):
        self.m_nom = m_nom           # номинальная масса, кг
        self.L = L                   # колёсная база, м
        self.g = g
        super().__init__(2, 5, 2)   # 2 состояния, 6 параметров, 2 входа


    def get_lateral_forces(self, state, theta, u):
        vy, r = state[0], state[1]
        vx, delta = u[0], u[1]

        # Извлекаем безразмерные параметры
        m_rel    = theta[0]
        Cf_rel   = theta[1]
        Cr_rel   = theta[2]
        a_rel    = theta[3]

        # Восстанавливаем физические величины
        m  = m_rel * self.m_nom
        Cf = Cf_rel * self.m_nom * self.g
        Cr = Cr_rel * self.m_nom * self.g
        a  = a_rel * self.L
        b  = self.L - a


        # Углы бокового увода (малые)
        alpha_f = delta - (vy + a * r) / vx
        alpha_r = -(vy - b * r) / vx

        # Боковые силы
        Fyf = Cf * alpha_f
        Fyr = Cr * alpha_r    
        return Fyf, Fyr
    
    def get_derivative(self, state, theta, u):
        r, vy = state[0], state[1]
        vx, delta = u[0], u[1]

        # Извлекаем безразмерные параметры
        m_rel    = theta[0]
        a_rel    = theta[3]
        Iz_rel   = theta[4]


        # Восстанавливаем физические величины
        m  = m_rel * self.m_nom
        a  = a_rel * self.L
        b  = self.L - a
        Iz = Iz_rel * m * self.L * self.L   # зависит и от m_rel, и от Iz_rel

        Fyf, Fyr = self.get_lateral_forces(state, theta, u)

        # Уравнения движения
        dvy = (Fyf + Fyr) / m - vx * r
        dr  = (a * Fyf - b * Fyr) / Iz

        return ca.vertcat(dr, dvy)
    
    def observation(self, state: SX, theta: SX, u: SX):
        Fyf, Fyr = self.get_lateral_forces(state, theta, u)
        m_rel    = theta[0]
        m  = m_rel * self.m_nom
        a_lat = (Fyf + Fyr)/m
        r, vy = state[0], state[1]
        return ca.vertcat(a_lat, r)
    
m = 2000


class DynamicModelRearAxle(ODESystem):
    def __init__(self, m, wheelbase, g=9.81):
        self.m = m
        self.wheelbase = wheelbase
        self.g = g
        self.delay = DelaySystem(order=2)
        # состояния:  wz, vy_rear
        super().__init__(2, 4, 2)

    def get_lateral_forces(self, rwa, vx, vy_rear, wz, theta):
        Cf_norm, Cr_norm, a_rel = theta[0], theta[1], theta[2]
        # Cf = ca.exp(Cf_norm * self.m * self.g)
        # Cr = ca.exp(Cr_norm * self.m * self.g)
        Cf = 6*Cf_norm * self.m * self.g
        Cr = 6*Cr_norm * self.m * self.g
        a = a_rel * self.wheelbase
        b = self.wheelbase - a
        # Скорость центра масс через заднюю ось
        vy_cm = vy_rear + b * wz
        alpha_f = rwa - (vy_cm + a * wz) / vx
        alpha_r = -(vy_cm - b * wz) / vx  
        Fyf = Cf * alpha_f
        Fyr = Cr * alpha_r
        return Fyf, Fyr, a, b

    def get_derivative(self, state, theta, u):
        wz, vy_rear = state[0], state[1]

        vx, rwa = u[0], u[1]

        # параметры
        a_rel = theta[2]
        Iz_norm = theta[3]


        a = a_rel * self.wheelbase
        b = self.wheelbase - a
        Iz = Iz_norm * self.m * self.wheelbase**2

        Fyf, Fyr, a_calc, b_calc = self.get_lateral_forces(rwa, vx, vy_rear, wz, theta)

        # динамика центра масс (необходима для сил и ускорения)
        vy_cm = vy_rear + b * wz
        dvy_cm = (Fyf + Fyr) / self.m - vx * wz
        dwz = (a * Fyf - b * Fyr) / Iz

        # производная vy_rear
        dvy_rear = dvy_cm - b * dwz
        return ca.vertcat(dwz, dvy_rear)
    
    def calc_acc(self, state, theta, u, d=0.0):
        """
        Вычисляет поперечное ускорение в точке, смещённой на d от центра масс.
        d > 0 – вперёд, к передней оси; d < 0 – назад.
        Состояние state (SX): [tau, psi, wz, vy_rear, rwa, rwa_dot, ...]
        """
        # Извлекаем компоненты
        wz, vy_rear = state[0], state[1]

        # Входы
        vx, rwa = u[0], u[1]

        # Параметры
        Cf_norm = theta[0]
        Cr_norm = theta[1]
        a_rel = theta[2]
        Iz_norm = theta[3]


        # Восстановление физических величин
        Cf = Cf_norm * self.m * self.g
        Cr = Cr_norm * self.m * self.g
        a = a_rel * self.wheelbase
        b = self.wheelbase - a
        Iz = Iz_norm * self.m * self.wheelbase * self.wheelbase

        Fyf, Fyr, a_calc, b_calc = self.get_lateral_forces(rwa, vx, vy_rear, wz, theta)
        a_lat_cm = (Fyf + Fyr) / self.m
        dr = (a * Fyf - b * Fyr) / Iz
        a_lat = a_lat_cm + d * dr
        return a_lat
    
    def observation(self, state: SX, theta: SX, u: SX):
        a_lat = self.calc_acc(state, theta, u, d = 0)
        wz, vy = state[0], state[1]
        return ca.vertcat(a_lat, wz)
# #
# config_dyn = {
#     "class": BicycleModelWithMass,
#     "args": [m, wheelbase],                                 # wheelbase
#     "c0": np.array([0.0]),      
#     "theta_true": np.array([1.0, 8.0, 7.5, 0.40, 0.24]),
#     "input_signal": get_input_signals,  #vx steering
# }

config_dyn = {
    "class": DynamicModelRearAxle,
    "args": [m, wheelbase],                                 # wheelbase
    "c0": np.array([0.0]),      
    "theta_true": np.array([2.0, 2.0, 0.4, 0.44]),
    "delta_theta": 0*np.array([8, 7.5, 0.4, 0.24]),
    "input_signal": get_input_signals,  #vx steering
}



In [11]:
measured_batches = [np.vstack((f_lat_ax(t), f_yaw_rate1(t))).T]
t_batches = [t]

system, c0, theta_init, _ = create_system(config_dyn)

In [12]:
measured_batches

[array([[-0.21      , -0.01456477],
        [-0.17999567, -0.0149968 ],
        [-0.14896627, -0.0154436 ],
        ...,
        [-0.9976109 , -0.07342865],
        [-1.0089047 , -0.07303113],
        [-1.02      , -0.07264061]], shape=(8000, 2), dtype=float32)]

In [13]:

import numpy as np
import matplotlib.pyplot as plt

class OptimizationConfig:
    """Конфигурация оптимизации для multiple shooting."""
    def __init__(self, n_obs):
        self.gamma = np.ones(n_obs)      # веса измерений
        self.gamma = np.array([0.1, 1.0])
        #assert len(self.gamma) == n_obs
        self.lambda_ = 0.001                # регуляризация Левенберга-Марквардта
        self.lambda_reg = 0.0000001               # дополнительная регуляризация (отключена)
        self.n_iter = 1                   # количество итераций
        self.c0_cost = 1.0                  # вес начальной точки в интервале
        self.mu = 5e-3                     # начальный параметр для метода с множителями
        self.mu_dec = 0.5
        self.n_shoot = 50

def setup_problem(system, config, state_measured_batches,
                  t_eval_batches):
    problem = MultipleShooting(system, N_shoot=config.n_shoot, gamma=config.gamma,
                               c0_cost=config.c0_cost, use_jax=True)
    for state_meas, t_meas in zip(state_measured_batches, t_eval_batches):
        problem.add_batch(state_meas, t_meas)
    return problem


if __name__ == "__main__":

    config = OptimizationConfig(system.n_obs)
    problem = setup_problem(system, config,
                            measured_batches,  t_batches)
    theta0 = theta_init   

    theta_full = problem.make_full_theta(theta0, n_iter = 10)
    # mu и lambda подбираются автоматически, ручной config больше не нужен
    theta_full, hist = run_optimization_adaptive(problem, theta_full)
    theta_hist = hist['theta']
    r_meas_hist, r_cont_hist = hist['r_meas'], hist['r_cont']
    ci_low_hist, ci_high_hist = hist['ci_low'], hist['ci_high']
    plot_solution(
        problem = problem, 
        theta_hist = theta_hist,
        plot_xy=1,
        plot_theta=True,
        plot_trajectory=0,
        plot_true_solution=False,
        plot_residuals=True,
        plot_measurements = 1,
        r_meas_hist=r_meas_hist,
        r_cont_hist=r_cont_hist,
        index=-1,
        theta_true=None,
        ci_low_hist=ci_low_hist,    # <-- добавить
        ci_high_hist=ci_high_hist,  # <-- добавить
        state_names=['a_y', 'w_z'],
        param_names=['Cf', 'Cr', 'a_rel', 'I_norm'],
    # param_names=[f'θ_{i}' for i in range( system.n_theta)]
    )

Solve batch 0
J_G_total shape: (98, 104), R_G_total len: 98
  J nnz: 95638, J_G nnz: 686
Iter   0 | time: 0.000s | R_meas: 1.922e-02 | R_cont: 4.010e-02 | mu: 5.00e-03
Solve batch 0
J_G_total shape: (98, 104), R_G_total len: 98


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

class OptimizationConfig:
    """Конфигурация оптимизации для multiple shooting."""
    def __init__(self, n_obs):
        self.gamma = np.ones(n_obs)      # веса измерений
        self.gamma = np.array([0.1, 1.0])
        #assert len(self.gamma) == n_obs
        self.lambda_ = 0.001                # регуляризация Левенберга-Марквардта
        self.lambda_reg = 0.0000001               # дополнительная регуляризация (отключена)
        self.n_iter = 1                   # количество итераций
        self.c0_cost = 1.0                  # вес начальной точки в интервале
        self.mu = 5e-3                     # начальный параметр для метода с множителями
        self.mu_dec = 0.5
        self.n_shoot = 50

def setup_problem(system, config, state_measured_batches,
                  t_eval_batches):
    problem = MultipleShooting(system, N_shoot=config.n_shoot, gamma=config.gamma,
                               c0_cost=config.c0_cost, use_jax=True)
    for state_meas, t_meas in zip(state_measured_batches, t_eval_batches):
        problem.add_batch(state_meas, t_meas)
    return problem


if __name__ == "__main__":

    config = OptimizationConfig(system.n_obs)
    problem = setup_problem(system, config,
                            measured_batches,  t_batches)
    theta0 = theta_init   

    theta_full = problem.make_full_theta(theta0, n_iter = 10)
    # mu и lambda подбираются автоматически, ручной config больше не нужен
    theta_full, hist = run_optimization_adaptive(problem, theta_full)
    theta_hist = hist['theta']
    r_meas_hist, r_cont_hist = hist['r_meas'], hist['r_cont']
    ci_low_hist, ci_high_hist = hist['ci_low'], hist['ci_high']
    plot_solution(
        problem = problem, 
        theta_hist = theta_hist,
        plot_xy=1,
        plot_theta=True,
        plot_trajectory=0,
        plot_true_solution=False,
        plot_residuals=True,
        plot_measurements = 1,
        r_meas_hist=r_meas_hist,
        r_cont_hist=r_cont_hist,
        index=-1,
        theta_true=None,
        ci_low_hist=ci_low_hist,    # <-- добавить
        ci_high_hist=ci_high_hist,  # <-- добавить
        state_names=['a_y', 'w_z'],
        param_names=['Cf', 'Cr', 'a_rel', 'I_norm'],
    # param_names=[f'θ_{i}' for i in range( system.n_theta)]
    )

Solve batch 0
J_G_total shape: (98, 104), R_G_total len: 98
  J nnz: 95638, J_G nnz: 686
Iter   0 | time: 0.000s | R_meas: 1.922e-02 | R_cont: 4.010e-02 | mu: 5.00e-03
Solve batch 0
J_G_total shape: (98, 104), R_G_total len: 98
Iter   1 | time: 0.000s | R_meas: 5.315e-02 | R_cont: 8.133e-02 | mu: 2.50e-03
Solve batch 0
J_G_total shape: (98, 104), R_G_total len: 98
Iter   2 | time: 0.000s | R_meas: 7.615e-03 | R_cont: 1.059e-02 | mu: 1.25e-03
Solve batch 0
J_G_total shape: (98, 104), R_G_total len: 98
Iter   3 | time: 0.000s | R_meas: 3.115e-02 | R_cont: 4.069e-01 | mu: 6.25e-04
Solve batch 0
J_G_total shape: (98, 104), R_G_total len: 98
Iter   4 | time: 0.000s | R_meas: 4.544e-03 | R_cont: 4.839e-02 | mu: 3.13e-04
Solve batch 0
J_G_total shape: (98, 104), R_G_total len: 98
Iter   5 | time: 0.000s | R_meas: 1.898e-03 | R_cont: 1.762e-03 | mu: 1.56e-04
Solve batch 0
J_G_total shape: (98, 104), R_G_total len: 98
Iter   6 | time: 0.000s | R_meas: 1.881e-03 | R_cont: 1.856e-06 | mu: 7.81e-0

In [14]:

plot_solution(
    problem = problem, 
    theta_hist = theta_hist,
    plot_xy=1,
    plot_theta=True,
    plot_trajectory=0,
    plot_true_solution=False,
    plot_residuals=True,
    plot_measurements = 1,
    r_meas_hist=r_meas_hist,
    r_cont_hist=r_cont_hist,
    index=-1,
    theta_true=None,
    fontsize = 20,
    # ci_low_hist=ci_low_hist,    # <-- добавить
    # ci_high_hist=ci_high_hist,  # <-- добавить
    state_names=['a_y', 'w_z'],
    param_names=['C<sub>f</sub>', 'C<sub>r</sub>', 'a<sub>rel</sub>', 'I<sub>norm</sub>']
   # param_names=[f'θ_{i}' for i in range( system.n_theta)]
)

In [31]:

plot_solution(
    problem = problem, 
    theta_hist = theta_hist,
    plot_xy=1,
    plot_theta=True,
    plot_trajectory=0,
    plot_true_solution=False,
    plot_residuals=True,
    plot_measurements = 1,
    r_meas_hist=r_meas_hist,
    r_cont_hist=r_cont_hist,
    index=0,
    theta_true=None,
    # ci_low_hist=ci_low_hist,    # <-- добавить
    # ci_high_hist=ci_high_hist,  # <-- добавить
    state_names=['a_y', 'w_z'],
    param_names=['C<sub>f</sub>', 'C<sub>r</sub>', 'a<sub>rel</sub>', 'I<sub>norm</sub>']
   # param_names=[f'θ_{i}' for i in range( system.n_theta)]
)

In [26]:
def theta_to_physical1(theta, m, L, g=9.81):

    Cf_norm, Cr_norm, a_rel, Iz_norm = theta

    Cf = Cf_norm * m * g
    Cr = Cr_norm * m * g
    a  = a_rel * L
    b  = L - a
    Iz = Iz_norm * m * L * L

    return {
        'Cf': Cf,
        'Cr': Cr,
        'a': a,
        'b': b,
        'Iz': Iz
    }
config = theta_to_physical1(theta_hist[-1][:4], m, wheelbase)
config, theta_hist[-1][:4]


({'Cf': np.float64(60121.15586674751),
  'Cr': np.float64(142566.83798045907),
  'a': np.float64(1.7854258227619848),
  'b': np.float64(1.1745741772380152),
  'Iz': np.float64(5466.977727312491)},
 array([3.0642791 , 7.26640357, 0.6031844 , 0.31198512]))

In [23]:
def theta_to_physical1(theta, m, L, g=9.81):

    Cf_norm, Cr_norm, a_rel, Iz_norm = theta

    Cf = Cf_norm * m * g
    Cr = Cr_norm * m * g
    a  = a_rel * L
    b  = L - a
    Iz = Iz_norm * m * L * L

    return {
        'Cf': Cf,
        'Cr': Cr,
        'a': a,
        'b': b,
        'Iz': Iz
    }
config = theta_to_physical1(theta_hist[-1][:4], m, wheelbase)
config


{'Cf': np.float64(73213.06401763306),
 'Cr': np.float64(115061.44555973665),
 'a': np.float64(1.5461694824351777),
 'b': np.float64(1.4138305175648223),
 'Iz': np.float64(5911.594306496334)}

In [24]:
def theta_to_physical_with_mass(theta, m_nom, L, g=9.81):

    m_rel, Cf_rel, Cr_rel, a_rel, Iz_rel = theta

    m  = m_rel * m_nom
    Cf = Cf_rel * m_nom * g
    Cr = Cr_rel * m_nom * g
    a  = a_rel * L
    b  = L - a
    Iz = Iz_rel * m * L * L

    return {
        'm': m,
        'Cf': Cf,
        'Cr': Cr,
        'a': a,
        'b': b,
        'Iz': Iz
    }
config = theta_to_physical_with_mass(theta_hist[-1][:5], m, wheelbase)
config


{'m': np.float64(7463.105404447814),
 'Cf': np.float64(115061.44555973665),
 'Cr': np.float64(10248.596366681819),
 'a': np.float64(0.998580119340597),
 'b': np.float64(1.961419880659403),
 'Iz': np.float64(12.126051992454691)}

In [17]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Optional

def plot_theta_history(
    theta_hist: List[np.ndarray],
    ci_low_hist: Optional[List[np.ndarray]] = None,
    ci_high_hist: Optional[List[np.ndarray]] = None,
    param_names: Optional[List[str]] = None,
    theta_true: Optional[np.ndarray] = None,
    theta_init: Optional[np.ndarray] = None,
    suptitle: str = "Parameter estimates"
):
    # Преобразуем списки в 2D-массивы
    theta_arr = np.array([np.asarray(t).flatten() for t in theta_hist])
    n_th, n_params = theta_arr.shape

    # Приводим CI к той же длине, что и theta (если CI заданы)
    if ci_low_hist is not None:
        ci_low_arr = np.array([np.asarray(c).flatten() for c in ci_low_hist])
        ci_high_arr = np.array([np.asarray(c).flatten() for c in ci_high_hist])
        # Обрезаем до минимального количества итераций
        min_len = min(n_th, len(ci_low_arr))
        if min_len < n_th:
            print(f"Внимание: длины не совпадают. Обрезаем до {min_len} точек.")
        theta_arr = theta_arr[:min_len]
        ci_low_arr = ci_low_arr[:min_len]
        ci_high_arr = ci_high_arr[:min_len]
    else:
        ci_low_arr = np.full_like(theta_arr, np.nan)
        ci_high_arr = np.full_like(theta_arr, np.nan)

    n_iter = theta_arr.shape[0]
    if param_names is None:
        param_names = [f"θ_{i}" for i in range(n_params)]

    fig, axes = plt.subplots(n_params, 1, figsize=(10, 2.2 * n_params), sharex=True)
    if n_params == 1:
        axes = [axes]

    iterations = np.arange(n_iter)

    for i, ax in enumerate(axes):
        if not np.all(np.isnan(ci_low_arr[:, i])):
            ax.fill_between(iterations, ci_low_arr[:, i], ci_high_arr[:, i],
                            alpha=0.2, color="C0", label="95% CI")
        ax.plot(iterations, theta_arr[:, i], color="C0", linewidth=1.5, label="Estimate")
        if theta_true is not None:
            ax.axhline(theta_true[i], color="green", linestyle="--", label="True")
        if theta_init is not None:
            ax.axhline(theta_init[i], color="red", linestyle=":", label="Init")
        ax.set_ylabel(param_names[i])
        ax.grid(True, alpha=0.3)
        ax.legend(loc="best", fontsize="small")
        ax.autoscale(axis="y", tight=False)
        ax.margins(y=0.1)

    ax.set_xlabel("Iteration")
    fig.suptitle(suptitle)
    fig.tight_layout()
    plt.show()

param_names = ["m_rel", "Cf_norm", "Cr_norm", "a_rel", "Iz_norm"]

theta_arr = np.array([np.asarray(t).flatten() for t in theta_hist])

plot_theta_history(theta_arr[:, :5], ci_low_hist, ci_high_hist,
                   param_names=param_names)

Внимание: длины не совпадают. Обрезаем до 10 точек.


<Figure size 1000x1100 with 5 Axes>

In [130]:
import jax.numpy as jnp
from jax import jacfwd

theta_opt = theta_hist[-1]      # последний вектор параметров
residuals = problem.residuals(theta_opt)
J = jacfwd(problem.residuals)(theta_opt)   # якобиан (N, 5)

# Весовая матрица (пример: равные веса для a_lat и r)
gamma = np.array([1.0, 1.0])
n_half = len(residuals) // 2
W_diag = jnp.concatenate([jnp.full(n_half, gamma[0]),
                          jnp.full(len(residuals)-n_half, gamma[1])])
W = jnp.diag(W_diag)

H = J.T @ W @ J
# Регуляризация для обращения
H_reg = H + 1e-8 * jnp.eye(len(theta_opt))
cov = jnp.linalg.inv(H_reg)
# Оценка дисперсии шума
ssq = residuals @ W @ residuals
sigma2 = ssq / (len(residuals) - len(theta_opt))
cov *= sigma2

SE = jnp.sqrt(jnp.diag(cov))
CV = np.asarray(SE / jnp.abs(theta_opt))

# Корреляции
corr = cov / jnp.outer(SE, SE)

# Сингулярные числа
eigvals = jnp.linalg.eigvalsh(H)

print("Число обусловленности:", np.linalg.cond(H))
print("Собственные значения:", eigvals)
print("Коэффициенты вариации:", CV)
print("Корреляционная матрица:\n", corr)

AttributeError: 'MultipleShooting' object has no attribute 'residuals'